# Milestone 1 — Official Benchmark on Google Colab

This notebook runs the **current** `arxiv-milestone1` project on a Google Colab **CPU-only** environment.

It is intentionally based on the current project structure:

```text
src/
├── main.py
├── benchmark.py
├── discover.py
├── download_sources.py
├── process_sources.py
├── metadata.py
└── references.py
```

> **Important:** This notebook does not use the old `crawler.py` / `run_colab_full_benchmark_resume.py` workflow.

## 1. Environment setup

The official benchmark should be run on Colab CPU-only. The benchmark measures real runtime, RAM, disk usage, crawl results, and reference statistics.

In [ ]:
from pathlib import Path
import os
import sys

PROJECT_DIR = Path('/content/milestone1')
BENCHMARK_DIR = Path('/content/benchmark-e2e-colab')
RESULTS_DIR = Path('/content/benchmark-results')

print('Python version:', sys.version)
print('Project directory:', PROJECT_DIR)
print('Benchmark directory:', BENCHMARK_DIR)
print('Results directory:', RESULTS_DIR)
print('\nEnvironment setup completed.')

In [ ]:
# Install the project's runtime dependencies.
%pip -q install requests psutil arxiv pandas matplotlib

## 2. Clone the team's GitHub repository

Run this cell **after the current project has been pushed to GitHub**.

Repository:
`https://github.com/ndkhoi2407/NMKHDL_Project_09`

In [ ]:
import shutil
import subprocess

REPO_URL = 'https://github.com/ndkhoi2407/NMKHDL_Project_09.git'

if PROJECT_DIR.exists():
    shutil.rmtree(PROJECT_DIR)

subprocess.run(
    ['git', 'clone', REPO_URL, str(PROJECT_DIR)],
    check=True
)

print('Repository cloned successfully.')
print('Project:', PROJECT_DIR)

In [ ]:
# Verify that this is the current project, not the old crawler-based project.
required_files = [
    PROJECT_DIR / 'src' / 'main.py',
    PROJECT_DIR / 'src' / 'benchmark.py',
    PROJECT_DIR / 'src' / 'discover.py',
    PROJECT_DIR / 'src' / 'download_sources.py',
    PROJECT_DIR / 'src' / 'process_sources.py',
    PROJECT_DIR / 'src' / 'metadata.py',
    PROJECT_DIR / 'src' / 'references.py',
    PROJECT_DIR / 'src' / 'requirements.txt',
]

missing = [str(p.relative_to(PROJECT_DIR)) for p in required_files if not p.is_file()]

if missing:
    raise FileNotFoundError('Missing current-project files:\n' + '\n'.join(missing))

print('Current project structure verified.')
for p in required_files:
    print('  OK:', p.relative_to(PROJECT_DIR))

## 3. Configure the Semantic Scholar API key safely

The current `references.py` reads `S2_API_KEY` from the environment. **Do not paste the key into a notebook cell.**

In Colab, add a secret named `S2_API_KEY` using the key icon in the left sidebar, then run the cell below.

In [ ]:
from google.colab import userdata

S2_API_KEY = userdata.get('S2_API_KEY')
if not S2_API_KEY:
    raise RuntimeError(
        'S2_API_KEY is not configured. Add it to Colab Secrets, then rerun this cell.'
    )

os.environ['S2_API_KEY'] = S2_API_KEY

# Only show that the secret exists; never print the key itself.
print('S2_API_KEY configured: YES')

## 4. Benchmark configuration

The assigned range is **2608.24007 → 2608.29006**, containing 5,000 arXiv IDs.

The benchmark output is kept outside the Git repository. This is important because the crawled corpus is large and must not be committed to the public source repository.

In [ ]:
PREFIX = '2608'
START = 24007
END = 29006

EXPECTED_PAPERS = END - START + 1

print(f'Assigned range: {PREFIX}.{START:05d} → {PREFIX}.{END:05d}')
print('Expected IDs:', EXPECTED_PAPERS)
print('Benchmark output:', BENCHMARK_DIR)

In [ ]:
# Show the benchmark CLI so we can verify that the notebook is using the current code.
%cd /content/milestone1
!python src/benchmark.py --help

## 5. Run the official end-to-end benchmark

**This is the long-running cell.** It runs the current `src/main.py` through `src/benchmark.py` and measures the complete pipeline.

Do not run another benchmark cell simultaneously.

The benchmark records:
- end-to-end wall time;
- time for discovery, source download, cleanup, metadata, and references;
- RAM peak and average;
- disk usage;
- paper/version success statistics;
- source size before/after image removal;
- reference statistics;
- Semantic Scholar metadata success.

In [ ]:
import shutil

# Start from a clean benchmark directory for the official measurement.
if BENCHMARK_DIR.exists():
    shutil.rmtree(BENCHMARK_DIR)
BENCHMARK_DIR.mkdir(parents=True, exist_ok=True)

%cd /content/milestone1

cmd = (
    f'python -u src/benchmark.py '
    f'--prefix {PREFIX} --start {START} --end {END} '
    f'--output-dir {BENCHMARK_DIR}'
)

print('Running official benchmark...')
print(cmd)
!{cmd}

## 6. Load the measured benchmark results

This section is for analysis only. It does not invent or estimate benchmark values; all values come from the files generated by the real run.

In [ ]:
import json
import pandas as pd

SUMMARY = BENCHMARK_DIR / 'data' / 'benchmark' / 'summary.json'
SAMPLES = BENCHMARK_DIR / 'data' / 'benchmark' / 'samples.csv'
PAPERS = BENCHMARK_DIR / 'data' / 'benchmark' / 'papers.csv'

for p in [SUMMARY, SAMPLES, PAPERS]:
    print(('OK  ' if p.is_file() else 'MISS'), p)

if not SUMMARY.is_file():
    raise FileNotFoundError('summary.json was not generated. Check the benchmark cell output.')

summary = json.loads(SUMMARY.read_text(encoding='utf-8'))
samples = pd.read_csv(SAMPLES) if SAMPLES.is_file() else pd.DataFrame()
papers = pd.read_csv(PAPERS) if PAPERS.is_file() else pd.DataFrame()

print('\nSummary loaded.')
print('Sample rows:', len(samples))
print('Paper rows:', len(papers))

In [ ]:
# Compact human-readable result table.
def first_existing(d, *keys):
    for k in keys:
        if k in d:
            return d[k]
    return None

result_rows = [
    ('Assigned papers', first_existing(summary, 'ids_in_range', 'papers_in_range')),
    ('Papers found', first_existing(summary, 'found', 'papers_found')),
    ('Versions discovered', first_existing(summary, 'total_versions', 'versions_discovered')),
    ('Versions downloaded', first_existing(summary, 'versions_downloaded')),
    ('PDF-only versions', first_existing(summary, 'versions_pdf_only')),
    ('Withdrawn 404 versions', first_existing(summary, 'versions_withdrawn_404')),
    ('Paper success rate', first_existing(summary, 'overall_success_rate', 'success_rate')),
    ('Download success rate', first_existing(summary, 'download_success_rate')),
    ('Metadata success rate', first_existing(summary, 'metadata_success_rate')),
    ('Reference success rate', first_existing(summary, 'references_success_rate')),
    ('Wall time (seconds)', first_existing(summary, 'wall_time_s', 'wall_time')),
    ('Average seconds / paper', first_existing(summary, 'avg_seconds_per_paper')),
    ('Peak RAM (bytes)', first_existing(summary, 'memory_peak_bytes', 'ram_peak_bytes')),
    ('Average RAM (bytes)', first_existing(summary, 'memory_avg_bytes', 'ram_avg_bytes')),
    ('Peak extra disk (bytes)', first_existing(summary, 'disk_peak_extra_bytes')),
    ('Final corpus (bytes)', first_existing(summary, 'final_corpus_bytes')),
    ('Average paper bytes before cleanup', first_existing(summary, 'avg_paper_bytes_before')),
    ('Average paper bytes after cleanup', first_existing(summary, 'avg_paper_bytes_after')),
    ('Average references / paper', first_existing(summary, 'avg_refs')),
]

results_df = pd.DataFrame(result_rows, columns=['Metric', 'Measured value'])
display(results_df)

## 7. Runtime by pipeline step

In [ ]:
steps = summary.get('steps', {})
if steps:
    step_df = pd.DataFrame(
        [{'Step': k, 'Seconds': v} for k, v in steps.items()]
    )
    display(step_df)
else:
    print('No step timing data found in summary.json.')

In [ ]:
import matplotlib.pyplot as plt

if steps:
    plt.figure(figsize=(9, 5))
    plt.bar(step_df['Step'], step_df['Seconds'])
    plt.ylabel('Seconds')
    plt.xlabel('Pipeline step')
    plt.title('Milestone 1 Runtime by Pipeline Step')
    plt.xticks(rotation=30, ha='right')
    plt.tight_layout()
    plt.show()

## 8. RAM and disk usage over time

In [ ]:
print('Sample columns:', list(samples.columns))
display(samples.head())

In [ ]:
if not samples.empty:
    time_col = next((c for c in ['elapsed_s', 'elapsed_seconds', 'seconds'] if c in samples.columns), None)
    ram_col = next((c for c in ['memory_bytes', 'ram_bytes', 'rss_bytes', 'pss_bytes'] if c in samples.columns), None)
    disk_col = next((c for c in ['disk_extra_bytes', 'disk_bytes', 'disk_usage_bytes'] if c in samples.columns), None)

    if time_col and ram_col:
        plt.figure(figsize=(9, 5))
        plt.plot(samples[time_col], samples[ram_col] / (1024**2))
        plt.xlabel('Elapsed time (seconds)')
        plt.ylabel('RAM (MiB)')
        plt.title('RAM Usage During Benchmark')
        plt.tight_layout()
        plt.show()

    if time_col and disk_col:
        plt.figure(figsize=(9, 5))
        plt.plot(samples[time_col], samples[disk_col] / (1024**3))
        plt.xlabel('Elapsed time (seconds)')
        plt.ylabel('Additional disk usage (GiB)')
        plt.title('Disk Usage During Benchmark')
        plt.tight_layout()
        plt.show()
else:
    print('No samples.csv data available.')

## 9. Paper-level validation

The table below is generated from the actual `papers.csv` produced by the benchmark.

In [ ]:
if papers.empty:
    print('No papers.csv data available.')
else:
    print('Columns:', list(papers.columns))
    display(papers.head())

    for col in ['download_status', 'metadata_status', 'refs_status', 'status']:
        if col in papers.columns:
            print(f'\n=== {col} ===')
            display(papers[col].value_counts(dropna=False).to_frame('count'))

## 10. Save only benchmark evidence

The public source repository should not contain the crawled corpus. This cell copies only the small benchmark evidence files (`summary.json`, `samples.csv`, `papers.csv`) to Google Drive for safekeeping.

In [ ]:
from google.colab import drive
import shutil

drive.mount('/content/drive', force_remount=False)

DRIVE_RESULTS = Path('/content/drive/MyDrive/Milestone1_Benchmark_Results')
DRIVE_RESULTS.mkdir(parents=True, exist_ok=True)

for src in [SUMMARY, SAMPLES, PAPERS]:
    if src.is_file():
        shutil.copy2(src, DRIVE_RESULTS / src.name)

print('Benchmark evidence saved to:', DRIVE_RESULTS)
for p in sorted(DRIVE_RESULTS.iterdir()):
    print(' ', p.name, '-', round(p.stat().st_size / 1024**2, 3), 'MiB')

## 11. Final benchmark checklist

- [ ] The current GitHub repository was cloned successfully.
- [ ] `src/benchmark.py` and `src/main.py` are the current versions.
- [ ] The assigned range is 2608.24007 → 2608.29006.
- [ ] The benchmark was run on Colab CPU-only.
- [ ] `S2_API_KEY` was supplied through Colab Secrets, not hard-coded.
- [ ] `summary.json`, `samples.csv`, and `papers.csv` were generated.
- [ ] Report tables/charts use the measured benchmark output.
- [ ] The crawled corpus was not added to the public Git repository.